# Week 13 — Block 2: Guided Demo (Visualizing ML and NLP Models)

**DATS 6401 · Visualization of Complex Data**

A trained model gives you a number: its accuracy. This demo asks the three questions that number cannot answer, and answers each with a picture.

1. **How does it fail?** Which errors, in which direction, and what they cost.
2. **What is it using?** Which inputs drive the predictions.
3. **How sure is it?** Whether its confidence tells you anything.

We ask them first of a model on a table (tumor measurements), then of a model on text (movie reviews). Part 4 looks at data with no labels at all, and Part 5 connects a client to the model server you built in Week 12.

Parts 1, 2 and 4 run offline. Part 3 downloads the IMDB reviews the first time it runs, and its last section downloads a pretrained model (about 270 MB) and needs `transformers` and `torch`. Part 5 needs your Week 12 server running.

## Part 1 — How well does it perform? (~18 min)

The data: 569 breast tumors, each described by 30 measurements of the cell nuclei in a biopsy image. Each tumor is labeled **malignant** or **benign**. It ships with scikit-learn, so nothing is downloaded.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

bc = load_breast_cancer(as_frame=True)
X, y = bc.data, bc.target                  # y: 0 = malignant, 1 = benign

print(X.shape)
print(pd.Series(y).map({0: "malignant", 1: "benign"}).value_counts())
X.iloc[:5, :6]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=150, random_state=42).fit(X_train, y_train)

print(f"Tumors in the test set: {len(y_test)}")
print(f"Accuracy: {model.score(X_test, y_test):.3f}")

95.8% sounds like the end of the story. It means 6 of 143 tumors were called wrong. The confusion matrix shows **which** six.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, display_labels=["malignant", "benign"], cmap="Blues")
plt.title("Confusion matrix: 143 test tumors")
plt.show()

Read it by row. Of the 53 malignant tumors, the model called 49 malignant and **4 benign**. Of the 90 benign tumors, it called 88 benign and **2 malignant**.

The two kinds of error do not cost the same. A benign tumor called malignant means more tests. A malignant tumor called benign means a cancer that goes untreated. Accuracy counts those six errors as equal; the matrix keeps them apart.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, model.predict(X_test),
                            target_names=["malignant", "benign"], digits=3))

**Recall** for malignant is 49 / 53 = 0.925: the share of real cancers the model caught. **Precision** is 49 / 51 = 0.961: the share of its "malignant" calls that were right. For this problem recall is the number to watch.

### The threshold is a choice

The model does not output a label. It outputs a probability, and the label comes from a cutoff of 0.5 that nobody chose on purpose. The ROC curve shows every cutoff at once.

In [ ]:
from sklearn.metrics import RocCurveDisplay

# Treat malignant as the class we are trying to catch
is_malignant = (y_test == 0)
p_malignant = model.predict_proba(X_test)[:, 0]

RocCurveDisplay.from_predictions(is_malignant, p_malignant, name="random forest")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="guessing")
plt.xlabel("False alarm rate (benign called malignant)")
plt.ylabel("Recall (malignant caught)")
plt.title("ROC curve: every threshold at once")
plt.legend()
plt.show()

In [ ]:
# What each threshold would have done to these 143 patients
rows = []
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    called = p_malignant >= t
    rows.append({"call malignant when P >=": t,
                 "missed cancers": int((is_malignant & ~called).sum()),
                 "false alarms": int((~is_malignant & called).sum())})
thresholds = pd.DataFrame(rows)
thresholds

In [ ]:
ax = thresholds.plot(x="call malignant when P >=", y=["missed cancers", "false alarms"],
                     marker="o", color=["firebrick", "steelblue"], figsize=(6.5, 4))
ax.invert_xaxis()
ax.set_xlabel("Threshold on P(malignant)  (lower = more cautious)")
ax.set_ylabel("Patients")
ax.set_title("Lowering the threshold trades missed cancers for false alarms")
plt.show()

At 0.5 the model misses 4 cancers and raises 2 false alarms. At 0.2 it misses none and raises 9. Which is better is not a modeling question. It depends on what a missed cancer and an extra biopsy cost, and the chart is what lets someone who is not a data scientist make that call.

## Part 2 — What is it using, and where does it fail? (~16 min)

### Feature importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values()

importance.tail(12).plot.barh(color="steelblue", figsize=(7, 4.5))
plt.xlabel("Importance (share of the forest's splits, weighted)")
plt.title("The 12 features the forest leans on most")
plt.show()

Sorted bars, so the comparison is on a common scale (Week 2). The top of the list is the "worst" size measurements: area, perimeter, radius.

Before believing the ranking, look at how those features relate to each other.

In [ ]:
top = ["worst area", "worst perimeter", "worst radius", "mean perimeter", "mean radius"]
X[top].corr().round(2)

They are almost the same measurement: radius, perimeter and area of the same nucleus. When features carry the same information, a forest splits the credit among them more or less arbitrarily.

**Permutation importance** asks a different question: shuffle one column, and see how much worse the model gets.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
compare = pd.DataFrame({"impurity importance": model.feature_importances_,
                        "permutation importance": perm.importances_mean},
                       index=X.columns).sort_values("impurity importance").tail(8)

compare.plot.barh(color=["steelblue", "orange"], figsize=(7, 4.5))
plt.xlabel("Importance")
plt.title("Two definitions of importance, same model")
plt.show()

Shuffling any single feature barely moves the accuracy, because its near-twins are still there to do the work. Neither chart is wrong. One shows what the model **split on**; the other shows what it **could not do without**. Say which one you are showing.

### The error map

Thirty features cannot be drawn directly. Project them to two dimensions (PCA here, as in Week 7) and mark the tumors the model got wrong.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=2).fit(scaler.transform(X_train))
pcs = pca.transform(scaler.transform(X_test))

wrong = model.predict(X_test) != y_test

fig, ax = plt.subplots(figsize=(7, 5))
for label, name, color in [(0, "malignant", "firebrick"), (1, "benign", "steelblue")]:
    keep = (y_test == label) & ~wrong
    ax.scatter(pcs[keep, 0], pcs[keep, 1], s=22, alpha=0.6, color=color, label=name)
ax.scatter(pcs[wrong, 0], pcs[wrong, 1], s=130, marker="x", color="black", linewidths=2.5,
           label="model was wrong")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Where the six errors are")
ax.legend()
plt.show()

The errors are not scattered. They sit where the two classes meet. The model is wrong about tumors that look like both.

### Does the model know when it is unsure?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(p_malignant[~wrong.to_numpy()], bins=20, range=(0, 1), color="lightgray", label="correct")
ax.hist(p_malignant[wrong.to_numpy()], bins=20, range=(0, 1), color="firebrick", label="wrong")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Predicted P(malignant)")
ax.set_ylabel("Test tumors")
ax.set_title("Most predictions are near 0 or 1; the errors are not")
ax.legend()
plt.show()

print("P(malignant) for the six errors:", np.sort(p_malignant[wrong.to_numpy()]).round(2))

For the six tumors it got wrong, the model said between 0.29 and 0.76. For nearly everything else it said close to 0 or 1. Its confidence carries real information, which is the reason your Week 12 server returns the probabilities and not only the label.

## Part 3 — The same three questions for a text model (~16 min, plus the pretrained model if time allows)

The data: IMDB movie reviews, each labeled positive or negative. We train on 20,000 and test on 1,000. The first run downloads the dataset from Hugging Face (about 80 MB); after that it is read from a local cache.

The model: TF-IDF turns each review into word weights (Week 7), and logistic regression learns one coefficient per word.

In [ ]:
from datasets import load_dataset

imdb = load_dataset("imdb")
train_df = pd.DataFrame(imdb["train"].shuffle(seed=42).select(range(20000)))
test_df = pd.DataFrame(imdb["test"].shuffle(seed=42).select(range(1000)))

print(train_df.shape, test_df.shape)
print(test_df["text"].iloc[0][:300], "...")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

text_model = make_pipeline(
    TfidfVectorizer(stop_words="english", max_features=5000),
    LogisticRegression(max_iter=1000, random_state=42),
).fit(train_df["text"], train_df["label"])

print(f"Accuracy: {text_model.score(test_df['text'], test_df['label']):.3f}")

ConfusionMatrixDisplay.from_estimator(
    text_model, test_df["text"], test_df["label"],
    display_labels=["negative", "positive"], cmap="Blues")
plt.title("Confusion matrix: 1,000 test reviews")
plt.show()

**How does it fail?** 141 of 1,000 wrong, split almost evenly: 71 negative reviews called positive and 70 the other way. Unlike the tumors, neither error is worse, and the matrix shows the model is not biased toward either class.

**What does the model receive?** A model cannot take text as input. The vectorizer keeps the 5,000 most common words and gives every review one number per word. Here is one short review and the numbers it becomes.

In [ ]:
vectorizer = text_model.named_steps["tfidfvectorizer"]
classifier = text_model.named_steps["logisticregression"]
vocabulary = vectorizer.get_feature_names_out()

review = test_df["text"][463]
print(review, "\n")

row = vectorizer.transform([review]).toarray()[0]      # one row, 5,000 numbers
print(f"Numbers in the row: {len(row)}.  Not zero: {(row > 0).sum()}")

pd.Series(row, index=vocabulary)[row > 0].sort_values().plot.barh(color="gray", figsize=(7, 4))
plt.xlabel("TF-IDF value of each word in this review")
plt.show()

The review uses 12 of the 5,000 words, so its row holds 12 values and 4,988 zeros. A word scores higher when it is rare in other reviews. The order of the words is not kept, which will matter later in this part.

**A picture that looks good and shows little.** The usual first picture of text is a word cloud. Draw one for each class.

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, label, name in [(axes[0], 0, "negative reviews"), (axes[1], 1, "positive reviews")]:
    text = " ".join(train_df.loc[train_df["label"] == label, "text"]).replace("<br />", " ")
    cloud = WordCloud(width=700, height=420, background_color="white",
                      random_state=42, max_words=80).generate(text)
    ax.imshow(cloud)
    ax.axis("off")
    ax.set_title(name)
plt.show()

The largest words in both clouds are "movie", "film" and "one". Size shows how often a word appears, and the most frequent words are the same in praise and in complaint. A word cloud cannot show what separates the two classes.

**What is it using?** The model can. A linear model on words has the most readable feature importance there is: one coefficient per word.

In [ ]:
words = pd.Series(classifier.coef_[0], index=vocabulary).sort_values()
show = pd.concat([words.head(12), words.tail(12)])

show.plot.barh(color=["firebrick"] * 12 + ["steelblue"] * 12, figsize=(7, 6))
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Coefficient  (negative review  <-   -> positive review)")
plt.title("The words that move the prediction most")
plt.show()

The model has learned a vocabulary of praise and complaint. That is also its weakness: it counts words and does not read sentences.

**Explain one prediction.** For a single review, what each word adds to the score is its TF-IDF value in that review times its coefficient. The model adds those contributions up.

In [ ]:
review = test_df["text"][743]
row = vectorizer.transform([review]).toarray()[0]
contribution = pd.Series(row * classifier.coef_[0], index=vocabulary)[row > 0].sort_values()

print("True label: positive.   Model: P(positive) =",
      round(text_model.predict_proba([review])[0, 1], 2))

contribution.plot.barh(color=np.where(contribution > 0, "steelblue", "firebrick"), figsize=(7, 5))
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("What each word adds to the score  (red: toward negative, blue: toward positive)")
plt.show()

The same numbers can be drawn on the text itself. This view is specific to text: you read the review and see what the model reacted to.

In [ ]:
import re
from IPython.display import HTML

strong = contribution.abs().max() / 2

def paint(match):
    word = match.group(0)
    c = contribution.get(word.lower())
    if c is None:
        return word                                   # a word the model ignores
    if c > 0:
        color = "steelblue" if c > strong else "lightsteelblue"
    else:
        color = "lightcoral" if -c > strong else "mistyrose"
    return f'<span style="background: {color}; padding: 0 3px">{word}</span>'

HTML('<p style="font-size: 16px; line-height: 1.8">' + re.sub(r"\w+", paint, review) + "</p>")

The model is right about this review, and part of its reason is wrong. The largest push toward positive comes from "great", which the writer used in "not great acting" and "not really a great film".

**Read the errors.** The fastest way to understand a text model is to read what it got wrong while being most sure of itself.

In [ ]:
p_positive = text_model.predict_proba(test_df["text"])[:, 1]
results = test_df.assign(p_positive=p_positive,
                         predicted=(p_positive >= 0.5).astype(int))
results["wrong"] = results["predicted"] != results["label"]
results["sureness"] = (results["p_positive"] - 0.5).abs()

confident_errors = (results[results["wrong"] & (results["text"].str.split().str.len() < 90)]
                    .sort_values("sureness", ascending=False))

for _, row in confident_errors.head(3).iterrows():
    truth = "positive" if row["label"] == 1 else "negative"
    print(f"TRUE: {truth}   MODEL: P(positive) = {row['p_positive']:.2f}")
    print(row["text"][:450], "\n")

The first review is sarcasm: "the most hilarious movie I've ever seen" about a film the writer is mocking. The second praises a film for being enjoyably bad, and uses "bad" twice. A model that adds up word weights cannot tell either from the opposite review. No metric would have told you that; reading three errors does.

**How sure is it?**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(results.loc[~results["wrong"], "p_positive"], bins=25, range=(0, 1), color="lightgray", label="correct")
ax.hist(results.loc[results["wrong"], "p_positive"], bins=25, range=(0, 1), color="firebrick", label="wrong")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Predicted P(positive)")
ax.set_ylabel("Test reviews")
ax.set_title("Errors pile up near 0.5")
ax.legend()
plt.show()

unsure = results["p_positive"].between(0.35, 0.65)
print(f"Reviews the model was unsure about (0.35 to 0.65): {unsure.mean():.0%} of all reviews")
print(f"Share of the errors that fall in that band:        {unsure[results['wrong']].mean():.0%}")

About a fifth of the reviews fall in the unsure band, and they account for more than half of the errors. An app built on this model should treat a 0.55 differently from a 0.95, for example by saying "mixed" or by asking a person.

**What word counts cannot see.** Give the model two pairs of sentences that mean opposite things.

In [ ]:
pairs = ["This movie is good.", "This movie is not good.",
         "It was boring.", "It was never boring."]

pd.DataFrame({"sentence": pairs,
              "word counts": text_model.predict_proba(pairs)[:, 1].round(2)})

In [ ]:
print("'good' is in the vocabulary: ", "good" in vectorizer.vocabulary_)
print("'not' is in the vocabulary:  ", "not" in vectorizer.vocabulary_)
print("'never' is in the vocabulary:", "never" in vectorizer.vocabulary_)

Each pair gets the same score. The model weighs every word on its own, so it could not connect "not" to "good" even if it saw both. Here it does not see "not" at all: `stop_words="english"` removed it, along with "never".

### A model that reads the sentence (if time allows)

DistilBERT is a small transformer. Other people have already trained it on sentences from movie reviews, so we download it and use it without training. It reads the words in order. This section needs `pip install transformers torch`, the same libraries as the Week 12 detection server, and downloads about 270 MB the first time.

In [ ]:
from transformers import pipeline

reader = pipeline("sentiment-analysis",
                  model="distilbert-base-uncased-finetuned-sst-2-english")

reader(pairs)

It gets all four right. Now run it on the same 1,000 test reviews. The pipeline returns a label and a score for that label, so the next cell turns each answer into P(positive) to match the first model. `truncation=True` is needed because the model reads at most 512 tokens and some reviews are longer. This takes about a minute.

In [ ]:
texts = test_df["text"].str.replace("<br />", " ").tolist()
answers = reader(texts, truncation=True, batch_size=16)

bert_p = np.array([a["score"] if a["label"] == "POSITIVE" else 1 - a["score"] for a in answers])
bert_wrong = (bert_p >= 0.5).astype(int) != test_df["label"]

print(f"Word counts: {1 - results['wrong'].mean():.3f}")
print(f"DistilBERT:  {1 - bert_wrong.mean():.3f}")
print()
print("The sarcastic review from earlier (a negative review):")
print(f"  word counts P(positive) = {results['p_positive'][108]:.2f}")
print(f"  DistilBERT  P(positive) = {bert_p[108]:.2f}")

DistilBERT is more accurate, 0.879 against 0.859, and it handles "not". The sarcastic review still fools it, and it is more sure of the wrong answer than the word-count model was.

**How sure is it?** Draw the same histogram as before.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(bert_p[~bert_wrong], bins=25, range=(0, 1), color="lightgray", label="correct")
ax.hist(bert_p[bert_wrong], bins=25, range=(0, 1), color="firebrick", label="wrong")
ax.set_xlabel("Predicted P(positive)")
ax.set_ylabel("Test reviews")
ax.set_title("DistilBERT is almost always sure")
ax.legend()
plt.show()

extreme = (bert_p > 0.95) | (bert_p < 0.05)
print(f"Reviews answered above 0.95 or below 0.05: {extreme.mean():.0%}")
print(f"Errors among those: {bert_wrong[extreme].sum()} of {bert_wrong.sum()}")

The word-count model's errors sat near 0.5, so its confidence was useful. DistilBERT answers near 0 or 1 for nine reviews in ten, and most of its errors are among those. It is the more accurate model, and its confidence tells you less. Which of the two you want in an app depends on whether the app uses the confidence.

**Inside the model (optional).** Before it answers, DistilBERT turns each review into 768 numbers. UMAP, from Week 7, draws them in two dimensions. This cell needs `pip install umap-learn`.

In [ ]:
import torch
import umap

inside = []
for start in range(0, len(texts), 16):
    batch = reader.tokenizer(texts[start:start + 16], truncation=True, max_length=512,
                             padding=True, return_tensors="pt").to(reader.model.device)
    with torch.no_grad():
        out = reader.model(**batch, output_hidden_states=True)
    inside.append(out.hidden_states[-1][:, 0].cpu().numpy())    # the 768 numbers
inside = np.concatenate(inside)

xy = umap.UMAP(random_state=42).fit_transform(inside)
labels = test_df["label"].to_numpy()

plt.figure(figsize=(7, 5))
plt.scatter(*xy[labels == 0].T, s=12, color="orange", label="negative review")
plt.scatter(*xy[labels == 1].T, s=12, color="steelblue", label="positive review")
plt.scatter(*xy[bert_wrong].T, s=42, marker="x", color="black", label="model was wrong")
plt.xticks([]); plt.yticks([])
plt.title("1,000 reviews, as DistilBERT represents them")
plt.legend()
plt.show()

The model puts the reviews in two groups, and its errors are reviews placed in the wrong group. This is the error map from Part 2, drawn from the model's own representation of the text.

## Part 4 — Structure without labels: clustering (~10 min)

Sometimes there is nothing to predict, and the question is whether the data has groups. Two methods you met in Week 7, on a shape where they disagree.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans, DBSCAN

pts, _ = make_moons(n_samples=400, noise=0.06, random_state=0)
labels_km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(pts)
labels_db = DBSCAN(eps=0.2, min_samples=5).fit_predict(pts)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].scatter(pts[:, 0], pts[:, 1], c=labels_km, cmap="coolwarm", s=18)
axes[0].set_title("K-means: splits by distance to two centers")
axes[1].scatter(pts[:, 0], pts[:, 1], c=labels_db, cmap="coolwarm", s=18)
axes[1].set_title("DBSCAN: follows the dense regions")
plt.show()

K-means assumes round groups and you tell it how many. DBSCAN finds groups as regions where points are packed closely, decides the number itself, and labels isolated points as **noise** instead of forcing them into a group.

That makes DBSCAN a natural fit for locations (Weeks 9 and 10). Here are 2,254 Tesla Supercharger sites in the lower 48 states.

In [ ]:
from pathlib import Path

local = Path("data/superchargers_us.csv")
URL = "https://gaimsds.github.io/dataviz/dats6401/weekly/week13/data/superchargers_us.csv"
sites = pd.read_csv(local if local.exists() else URL)

# Latitude and longitude are angles, so measure distance on the sphere (haversine)
EARTH_RADIUS_KM = 6371.0088
eps_km = 25

sites["cluster"] = DBSCAN(eps=eps_km / EARTH_RADIUS_KM, min_samples=5,
                          algorithm="ball_tree", metric="haversine"
                          ).fit_predict(np.radians(sites[["lat", "lon"]]))

clustered = sites[sites["cluster"] >= 0]
noise = sites[sites["cluster"] == -1]
print(f"{clustered['cluster'].nunique()} clusters, {len(noise)} sites labeled noise")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(noise["lon"], noise["lat"], s=5, color="lightgray", label="noise (no dense neighborhood)")
ax.scatter(clustered["lon"], clustered["lat"], s=9, c=clustered["cluster"], cmap="tab20")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"DBSCAN: at least 5 sites within {eps_km} km of each other")
ax.legend(loc="lower left")
plt.show()

(clustered.groupby("cluster")
          .agg(sites=("cluster", "size"), main_city=("City", lambda s: s.mode()[0]),
               state=("State", lambda s: s.mode()[0]))
          .sort_values("sites", ascending=False).head(6))

The clusters are metropolitan areas, found without telling the algorithm how many cities to look for. The gray sites are the highway network between them.

The result depends on the two parameters, and there is no correct value to find.

In [ ]:
rows = []
for km in [10, 25, 50, 100]:
    labels = DBSCAN(eps=km / EARTH_RADIUS_KM, min_samples=5, algorithm="ball_tree",
                    metric="haversine").fit_predict(np.radians(sites[["lat", "lon"]]))
    found = pd.Series(labels)
    rows.append({"radius (km)": km,
                 "clusters": found[found >= 0].nunique(),
                 "noise sites": int((found == -1).sum()),
                 "largest cluster": int(found[found >= 0].value_counts().iloc[0])})
pd.DataFrame(rows)

At 100 km most of the eastern United States, from Florida to Massachusetts, merges into one cluster of 1,093 sites. A clustering chart should state its parameters the way a histogram should state its bin width (Week 7: the parameters are part of the result).

## Part 5 — The live loop (~10 min)

Everything above looked at a model from the inside. Your users will only see it through an API. This part makes this notebook a client of the server you built in Week 12.

1. Terminal 1, in your Week 12 folder: start your completed server, for example `uvicorn server_skeleton:app --reload` or `flask --app server_flask run --port 8000`.
2. Run the next cell.
3. Then `streamlit run app_live_predictions.py`: the same call, with sliders and the full distribution. Stop the server while the app is open and watch what the client does.

In [ ]:
import requests

# A flower on the versicolor / virginica boundary
payload = {"sepal_length": 6.0, "sepal_width": 2.7, "petal_length": 5.0, "petal_width": 1.7}
try:
    r = requests.post("http://127.0.0.1:8000/predict", json=payload, timeout=5)
    r.raise_for_status()
    out = r.json()
    print("LIVE prediction:", out["prediction"], f"(confidence {out['confidence']:.2f})")
    pd.Series(out["probabilities"]).plot.barh(color="steelblue", figsize=(6, 2.2), xlim=(0, 1))
    plt.title("The full distribution the server returned")
    plt.show()
except requests.RequestException as e:
    print("Server not reachable:", e)
    print("-> start your Week 12 server first (see the steps above)")

The server answers 0.53 for this flower. Parts 2 and 3 showed why that number matters: errors live where the model is unsure. A client that shows only the label hides exactly the cases a user most needs to see.